***BÀI 1***

In [16]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score, classification_report
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, RobustScaler, StandardScaler

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")
np.random.seed(42)          # cố định ngẫu nhiên -> kết quả tái lập được
print("Sẵn sàng.")

Sẵn sàng.


In [17]:
try:
    df = sns.load_dataset("titanic")
    print("Đã tải từ seaborn.")
except Exception:
    url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
    df = pd.read_csv(url)
    df.columns = [c.lower() for c in df.columns]
    print("Đã tải từ URL.")
df.head()

Đã tải từ seaborn.


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [18]:
# TODO 1a: in tỷ lệ missing của tất cả các cột

print(round(df.isnull().mean(),4))

# TODO 1b: bỏ các cột rò rỉ/dư thừa, gán lại vào biến df
leaky = ['alone', 'embark_town', 'deck', 'who', 'class', 'alive', 'adult_male' ]      # điền danh sách cột cần bỏ (chỉ những cột có trong df)
# df = df.drop(columns=...)
df = df.drop(columns = leaky)

# print("Các cột còn lại:", list(df.columns))
print("\nCác cột còn lại:", list(df.columns))

survived       0.0000
pclass         0.0000
sex            0.0000
age            0.1987
sibsp          0.0000
parch          0.0000
fare           0.0000
embarked       0.0022
class          0.0000
who            0.0000
adult_male     0.0000
deck           0.7722
embark_town    0.0022
alive          0.0000
alone          0.0000
dtype: float64

Các cột còn lại: ['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']


In [19]:
# TODO 2: shape, info, describe
print(df.shape)
print("Biến mục tiêu là survived: là biến mà mô hình cần phải dự đoán từ những dữ kiện của biến khác\n")
print(df.info())
print(df.describe())
df.describe(include="object")

(891, 8)
Biến mục tiêu là survived: là biến mà mô hình cần phải dự đoán từ những dữ kiện của biến khác

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   survived  891 non-null    int64  
 1   pclass    891 non-null    int64  
 2   sex       891 non-null    str    
 3   age       714 non-null    float64
 4   sibsp     891 non-null    int64  
 5   parch     891 non-null    int64  
 6   fare      891 non-null    float64
 7   embarked  889 non-null    str    
dtypes: float64(2), int64(4), str(2)
memory usage: 55.8 KB
None
         survived      pclass         age       sibsp       parch        fare
count  891.000000  891.000000  714.000000  891.000000  891.000000  891.000000
mean     0.383838    2.308642   29.699118    0.523008    0.381594   32.204208
std      0.486592    0.836071   14.526497    1.102743    0.806057   49.693429
min      0.000000    1.000000    0.42

C:\Users\Admin\AppData\Local\Temp\ipykernel_91936\3994035649.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df.describe(include="object")


,sex,embarked
count,891,889
unique,2,3
top,male,S
freq,577,644


In [20]:
# TODO 3: bảng missing (count + %)
missing_counts = df.isnull().sum()
missing_percent = (missing_counts/len(df)*100).round(4)
print(f"{missing_counts}\n{missing_percent}")

survived      0
pclass        0
sex           0
age         177
sibsp         0
parch         0
fare          0
embarked      2
dtype: int64
survived     0.0000
pclass       0.0000
sex          0.0000
age         19.8653
sibsp        0.0000
parch        0.0000
fare         0.0000
embarked     0.2245
dtype: float64


In [21]:
median = df['age'].median()
df['age'] = df['age'].fillna(median)

mode = df['embarked'].mode()[0]
df['embarked'] = df['embarked'].fillna(mode)


In [22]:
# TODO 4: đếm outlier theo IQR và Z-score cho 'age' và 'fare'
def dem_outlier_iqr(s):
    s = s.dropna()
    Q1 = s.quantile(0.25)
    Q3 = s.quantile(0.75)
    iqr = Q3 - Q1
    return ((s < Q1 - 1.5*iqr) | (s> Q3 + 1.5 * iqr)).sum()
def dem_outlier_zscore(s, nguong=3.0):
    s =s.dropna()
    z_score = np.abs(stats.zscore(s))
    return (z_score > nguong).sum()

# for col in ["age", "fare"]:
for col in ["age", "fare"]:
    print(f"{col}:\nSố outlier theo IQR: {dem_outlier_iqr(df[col])}\nSố outlier theo z_score: {dem_outlier_zscore(df[col])}\n")

age:
Số outlier theo IQR: 66
Số outlier theo z_score: 7

fare:
Số outlier theo IQR: 116
Số outlier theo z_score: 20



In [23]:
# TODO 6: chia train/val/test có stratify
X = df.drop(columns=['survived'])
y = df['survived']

# X_tmp, X_test, y_tmp, y_test = train_test_split(...)
X_tmp, X_test, y_tmp, y_test = train_test_split(X, y, test_size=0.15, stratify=y, random_state= 1)

# X_train, X_val, y_train, y_val = train_test_split(...)
X_train, X_val, y_train, y_val = train_test_split(X_tmp, y_tmp, test_size= 15/85, stratify=y_tmp, random_state=1)

# print("Train/Val/Test:", ...)
print(f"Shape:\nTrain: {X_train.shape}\nValidation: {X_val.shape}\nTest: {X_test.shape}")

# in tỷ lệ survived từng tập
print(f"\nTỉ lệ survived của từng tập:\nTrain: {y_train.sum()/len(y_train)*100:.3f}%\nValidation: {y_val.sum()/len(y_val)*100:.3f}%\nTest: {y_test.sum()/len(y_test)*100:.3f}%")

Shape:
Train: (623, 7)
Validation: (134, 7)
Test: (134, 7)

Tỉ lệ survived của từng tập:
Train: 38.363%
Validation: 38.806%
Test: 38.060%


In [24]:
num_cols = ["age", "sibsp", "parch", "fare"]
cat_cols = ["sex", "embarked"]
ord_cols = ["pclass"]

# TODO 7: xây pipeline cho biến số và biến phân loại
pipe_so  = Pipeline([
    # ("imputer", ...),
    # ("scaler",  ...),
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler())
])
pipe_cat = Pipeline([
    # ("imputer", ...),
    # ("onehot",  ...),
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('scaler', OneHotEncoder())
])

preprocess = ColumnTransformer([
    # ("num", pipe_so,  num_cols),
    # ("cat", pipe_cat, cat_cols),
    # ("ord", "passthrough", ord_cols),
    ("num", pipe_so, num_cols),
    ("cat", pipe_cat, cat_cols),
    ("ord", "passthrough", ord_cols)
])

# preprocess.fit(X_train)               # fit CHỈ trên train
# X_train_t = preprocess.transform(X_train)
# ... transform cho val, test
# print(X_train_t.shape, list(preprocess.get_feature_names_out()))
preprocess.fit(X_train)
X_train_t = preprocess.transform(X_train)
X_val_t = preprocess.transform(X_val)
X_test_t = preprocess.transform(X_test)
print(X_train_t.shape, list(preprocess.get_feature_names_out()))


(623, 10) ['num__age', 'num__sibsp', 'num__parch', 'num__fare', 'cat__sex_female', 'cat__sex_male', 'cat__embarked_C', 'cat__embarked_Q', 'cat__embarked_S', 'ord__pclass']


*LOGISTIC REGRESSION*

In [25]:
logistic_model = LogisticRegression()
logistic_model.fit(X_train_t, y_train)
y_pred_log = logistic_model.predict(X_test_t)

*LINEAR REGRESSION*

In [26]:
linear_model = LinearRegression()
linear_model.fit(X_train_t, y_train)
y_pred_lin = linear_model.predict(X_test_t)
y_pred_lin_class = (y_pred_lin >= 0.5).astype(int)

In [27]:
print("LOGISTIC REGRESSION")
print(f"Accuracy: {accuracy_score(y_test, y_pred_log):.4f}")
print(classification_report(y_test, y_pred_log))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_log))

print("\nLINEAR REGRESSION")
print(f"Accuracy: {accuracy_score(y_test, y_pred_lin_class):.4f}")
print(classification_report(y_test, y_pred_lin_class))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_lin_class))

LOGISTIC REGRESSION
Accuracy: 0.7836
              precision    recall  f1-score   support

           0       0.79      0.88      0.83        83
           1       0.76      0.63      0.69        51

    accuracy                           0.78       134
   macro avg       0.78      0.75      0.76       134
weighted avg       0.78      0.78      0.78       134

Confusion Matrix:
 [[73 10]
 [19 32]]

LINEAR REGRESSION
Accuracy: 0.7910
              precision    recall  f1-score   support

           0       0.80      0.88      0.84        83
           1       0.77      0.65      0.70        51

    accuracy                           0.79       134
   macro avg       0.78      0.76      0.77       134
weighted avg       0.79      0.79      0.79       134

Confusion Matrix:
 [[73 10]
 [18 33]]


Nhận xét và đánh giá

-Tổng quan hiệu suất của cả hai mô hình khá tương đồng, chỉ chênh lệch rất ít (0.0074 ở accuracy).

-Mô hình nhạy với nhóm người thiệt mạng, khi chỉ số recall của lớp ở cả hai mô hình đều khá cao: 0.88, nghĩa là mô hình dự đoán những trường hợp đã mất mạng thì tỉ lệ họ mất mạng thật khá cao. Trong khi đó recall của lớp sống sót ở cả hai mô hình lại thấp:  với 0.63 ở logistic và 0.65 ở linear, mô hình đang bỏ sót tới hơn 35% số người sống sót thực tế, cho rằng họ đã thiệt mạng.

-Nếu nhìn vào những con số thì dường như linear regression chỉ nhỉnh hơn một ít. Nhưng trong thực tế, logistic regression vẫn nên là sự ưu tiên trong các bài toán phân loại vì linear là mô hình dự đoán một giá trị liên tục, đầu ra sẽ là một đường thẳng qua các điểm dữ liệu và các giá trị dự đoán có thể nằm ngoài khoảng [0,1]. Còn logistic sử dụng hàm sigmoid nên bảo đảm giá trị xác suất luôn nằm trong khoảng [0,1].

***BÀI 2***

In [28]:
from sklearn.neighbors import KNeighborsClassifier


train_df = pd.read_csv('Dry_Bean_Dataset/dry_bean_train.csv')
test_df = pd.read_csv('Dry_Bean_Dataset/dry_bean_test.csv')

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Train shape: (10834, 17)
Test shape: (2709, 17)


**LOGISTIC REGRESSION**

In [29]:
X_train = train_df.drop(columns='class')
y_train = train_df['class']

X_test = test_df.drop(columns='class')
y_test = test_df['class']

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

logistic_model = LogisticRegression(random_state=42)
logistic_model.fit(X_train_scaled, y_train)
y_pred_log = logistic_model.predict(X_test_scaled)

print(f"Accuracy: {accuracy_score(y_test, y_pred_log):.4f}")
print(classification_report(y_test, y_pred_log))



Accuracy: 0.9192
              precision    recall  f1-score   support

    BARBUNYA       0.93      0.89      0.91       265
      BOMBAY       1.00      1.00      1.00       104
        CALI       0.91      0.94      0.93       326
    DERMASON       0.93      0.91      0.92       709
       HOROZ       0.96      0.94      0.95       372
       SEKER       0.93      0.94      0.94       406
        SIRA       0.86      0.88      0.87       527

    accuracy                           0.92      2709
   macro avg       0.93      0.93      0.93      2709
weighted avg       0.92      0.92      0.92      2709



**KNN**

In [30]:
knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train_scaled, y_train)
y_pred_knn = knn_model.predict(X_test_scaled)


print(f"Accuracy: {accuracy_score(y_test, y_pred_knn):.4f}")
print(classification_report(y_test, y_pred_knn))


#Sử dụng GridSearchCV để lựa chọn giá trị K phù hợp thông qua Cross Validation
from sklearn.model_selection import GridSearchCV

param_grid = {'n_neighbors': [3, 5, 7, 9, 11, 15, 17, 21, 25]}
grid_search = GridSearchCV(KNeighborsClassifier(), param_grid, cv=5, scoring='accuracy')
grid_search.fit(X_train_scaled, y_train)

print(f"Số láng giềng K tối ưu nhất: {grid_search.best_params_}")

Accuracy: 0.9155
              precision    recall  f1-score   support

    BARBUNYA       0.93      0.88      0.90       265
      BOMBAY       1.00      1.00      1.00       104
        CALI       0.90      0.95      0.92       326
    DERMASON       0.92      0.91      0.91       709
       HOROZ       0.96      0.93      0.95       372
       SEKER       0.95      0.94      0.94       406
        SIRA       0.85      0.87      0.86       527

    accuracy                           0.92      2709
   macro avg       0.93      0.93      0.93      2709
weighted avg       0.92      0.92      0.92      2709

Số láng giềng K tối ưu nhất: {'n_neighbors': 17}


*Nhận xét*

-Cả hai mô hình Logistic Regression và KNN có độ chính xác khá tương đồng: lần lượt là 91.92% và 91.55%.

-Cả hai mô hình đều phân loại hoàn hảo hạt BOMBAY với các giá trị precision, recall và f1-score đều bằng 1.

-Tỉ lệ dự đoán đúng của hạt SIRA đều là thấp nhất ở cả hai model.